In [ ]:
import psycopg2
import clickhouse_connect
#!pip install clickhouse-connect

In [7]:
#Connect to PostgreSQL
pg_conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="pagila",
    user="admin",
    password="admin123"
)

print("Connected to PostgreSQL")

Connected to PostgreSQL


In [9]:
# Test actual data
cursor = pg_conn.cursor()

cursor.execute("""
    SELECT
        customer_id,
        first_name,
        last_name
    FROM public.customer
    LIMIT 5;
""")

rows = cursor.fetchall()

rows

[(1, 'MARY', 'SMITH'),
 (2, 'PATRICIA', 'JOHNSON'),
 (3, 'LINDA', 'WILLIAMS'),
 (4, 'BARBARA', 'JONES'),
 (5, 'ELIZABETH', 'BROWN')]

In [18]:
pg_cursor = pg_conn.cursor()
pg_cursor.execute("SELECT version();")
pg_cursor.fetchone()

('PostgreSQL 17.10 (Debian 17.10-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)

In [19]:
pg_cursor.execute("""
    SELECT
        customer_id,
        store_id,
        first_name,
        last_name,
        email,
        address_id,
        activebool,
        create_date,
        last_update,
        active
    FROM public.customer
    ORDER BY customer_id;
""")

customers = pg_cursor.fetchall()

len(customers)

600

In [20]:
# Connect to ClickHouse
ch_client = clickhouse_connect.get_client(
    host="localhost",
    port=8123,
    username="default",
    password="clickhouse"
)

print("Connected to ClickHouse")

Connected to ClickHouse


In [13]:
ch_client.query("SELECT version()").result_rows

[('26.9.1.1629',)]

In [14]:
ch_client.query("""
    SELECT count(*)
    FROM pagila.customer
""").result_rows

[(0,)]

In [17]:
"""
┌───────────────┐
│   PostgreSQL  │
│   localhost   │
│     :5432     │
└───────┬───────┘
        │
        │ psycopg2
        ▼
┌───────────────┐
│     Python    │
│    Notebook   │
└───────┬───────┘
        │
        │ clickhouse-connect
        ▼
┌───────────────┐
│   ClickHouse  │
│   localhost   │
│     :8123     │
└───────────────┘

"""

'\n┌───────────────┐\n│   PostgreSQL  │\n│   localhost   │\n│     :5432     │\n└───────┬───────┘\n        │\n        │ psycopg2\n        ▼\n┌───────────────┐\n│     Python    │\n│    Notebook   │\n└───────┬───────┘\n        │\n        │ clickhouse-connect\n        ▼\n┌───────────────┐\n│   ClickHouse  │\n│   localhost   │\n│     :8123     │\n└───────────────┘\n\n'

In [21]:
ch_client.insert(
    "pagila.customer",
    customers,
    column_names=[
        "customer_id",
        "store_id",
        "first_name",
        "last_name",
        "email",
        "address_id",
        "activebool",
        "create_date",
        "last_update",
        "active"
    ]
)

In [22]:
ch_client.query("""
    SELECT count(*)
    FROM pagila.customer
""").result_rows

[(600,)]